# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced by `@id` fields throughout to ensure reproducibility and data provenance.

In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier (@id): {metadata.id}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Explore available record sets, fields, and columns as defined by their `@id` fields.

In [ ]:
# List all record set @ids available in the dataset
from pprint import pprint

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available Record Sets and their @id fields:")
    for rs in record_sets:
        print(f"- Name: {rs.name}, @id: {rs.id}")

# For illustration, list all fields (by @id) in each record set
record_set_fields = {}
for rs in record_sets:
    field_ids = [field.id for field in rs.fields]
    record_set_fields[rs.id] = field_ids
    print(f"Record Set '@id': {rs.id}")
    print("  Field @ids:")
    for fid in field_ids:
        print(f"    - {fid}")

## 3. Data Extraction
Extract data from a specific record set into a DataFrame for further analysis. We will use the record set and field `@id` shown above.

_Note: If the dataset contains only one record set, it will be selected automatically; otherwise, choose from the list above._

In [ ]:
# Prepare extraction by referencing by @id
# Select the first record set for demonstration
record_sets = dataset.record_sets
if not record_sets:
    raise RuntimeError('No record sets defined in the dataset package.')

selected_rs = record_sets[0]
selected_rs_id = selected_rs.id
print(f"Selected Record Set: {selected_rs.name} (@id: {selected_rs_id})")

# Extract records into a DataFrame
records = list(dataset.records(record_set=selected_rs_id))
df = pd.DataFrame(records)

print("Fields (@id) as DataFrame columns:")
pprint(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Here we demonstrate fundamental EDA techniques—such as filtering, normalization, categorization, and grouping—referencing fields by their `@id`.

Let's inspect the DataFrame for suitable numeric and categorical fields:

In [ ]:
# Preview schema to pick sample numeric/categorical fields by @id
print("Field @ids:")
for field in selected_rs.fields:
    print(f"{field.name}: {field.id} | Type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")

# Suppose 'Age' is a numeric field (adjust with real @id as needed):
# For this example we'll try to infer a candidate @id for 'Age' or fallback to the first numeric-looking column.

import re

def guess_numeric_field(fields):
    for field in fields:
        # Common id or name heuristics
        if re.search(r'age', field.id, re.IGNORECASE) or re.search(r'age', field.name, re.IGNORECASE):
            return field.id
    # Fallback: pick the first field as default
    return fields[0].id

numeric_field_id = guess_numeric_field(selected_rs.fields)
print(f"Using numeric field for EDA: {numeric_field_id}")

# Similarly, try to guess a grouping field
def guess_group_field(fields):
    for field in fields:
        if re.search(r'sex|gender|group', field.id, re.IGNORECASE):
            return field.id
    # Fallback
    return fields[-1].id

group_field_id = guess_group_field(selected_rs.fields)
print(f"Using group field: {group_field_id}")

In [ ]:
# Filtering, normalization, and groupby demonstration (with @id references)
field = numeric_field_id

# Coerce column to numeric (ignore errors), typical in clinical tabular data
df[field] = pd.to_numeric(df[field], errors='coerce')
print(f"{field} value statistics:")
print(df[field].describe())

threshold = df[field].quantile(0.5)  # median threshold
filtered_df = df[df[field] > threshold]
print(f"Filtered records where {field} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{field}_normalized"] = (filtered_df[field] - filtered_df[field].mean()) / filtered_df[field].std()
print(f"Normalized {field} for filtered records:")
print(filtered_df[[field, f"{field}_normalized"]].head())

# Group by another field (e.g. sex/gender/etc. by @id)
group_field = group_field_id
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[field].mean().to_frame(name=f"mean_{field}")
    print(f"Grouped data by {group_field} (mean {field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields. In all plots, axes and labels use the field's `@id` reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[field].dropna(), bins=10, kde=True)
plt.xlabel(field)
plt.title(f"Distribution of {field} (@id)")
plt.show()

# Boxplot by group field (if available)
if group_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field, y=field)
    plt.xlabel(group_field)
    plt.ylabel(field)
    plt.title(f"{field} by {group_field} (using @id)")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded, explored, and visualized the FAIR² clinical dataset, referencing all records and fields by their Croissant `@id`. This approach ensures full traceability to the data's formal schema and enables consistent data science workflows. Typical EDA tasks—including filtering, normalization, and grouping—have been demonstrated programmatically using these unique identifiers.You may now proceed to more advanced analysis or model building using the records and `@id`-referenced features extracted via `mlcroissant`.